# 初始化 client（连接 DeepSeek API）

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

# tool描述，给LLM看的

In [ ]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的当前天气信息",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 北京、上海、杭州"
                    }
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_time",
            "description": "查询指定城市当地的时间",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，例如 北京、上海、杭州"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

# tool的执行函数，返回的结果给LLM看

In [ ]:
import json
def get_weather(city: str) -> str:
    fake_data = {
        "北京": {"temp": 18, "condition": "晴"},
        "上海": {"temp": 22, "condition": "小雨"},
        "杭州": {"temp": 20, "condition": "多云"},
    }
    if city in fake_data:
        return json.dumps(fake_data[city], ensure_ascii=False)
    else:
        return json.dumps({"error": f"没有 {city} 的数据"}, ensure_ascii=False)

# 测一下函数本身
print(get_weather("杭州"))
print(get_weather("纽约"))

def get_time(city: str) -> str:
    fake_times = {
        "北京": "14:30",
        "上海": "14:30",
        "杭州": "14:30",
        "纽约": "02:30",
    }
    return json.dumps({"time": fake_times.get(city, "未知")}, ensure_ascii=False)

# 测一下函数本身能用
print(get_time("杭州"))




In [ ]:
#用户给出问题
messages = [
    {"role": "user", "content": "杭州现在几点了,什么天气"}
]
#第一次API调用
resp = client.chat.completions.create(
    model="deepseek-chat",
    messages=messages,
    tools=tools
)
#将LLM返回的调用工具请求添加到LLM的“记忆”中
assistant_message=resp.choices[0].message
messages.append(assistant_message)
#创造一个工具对照表，快速查询获得工具
function_registry = {
    "get_time" : get_time ,
    "get_weather" : get_weather
}
#将LLM需要的工具依次添加到他的“记忆”里
for tc in resp.choices[0].message.tool_calls:
    tool_call = tc
    function_name = tc.function.name
    function_args = json.loads(tc.function.arguments)
    messages.append({
    "role": "tool",
    "tool_call_id": tc.id,
    "content": function_registry[function_name](**function_args)
    
})
#第二次API调用，LLM给出答案
final_resp = client.chat.completions.create(
     model="deepseek-chat",
     messages=messages,
     tools=tools
 )

print(final_resp.choices[0].message.content)